# Hinglish Smart Writing Assistant — Colab backend

This notebook runs the GPU inference service used by the local Streamlit app. It loads Param-1 and IndicTrans2, while transliteration uses AI4Bharat’s hosted IndicXlit API.

Before continuing: open the [IndicTrans2 model page](https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M), log in, and accept its access conditions. In Colab select **Runtime → Change runtime type → T4 GPU** (or better).

In [ ]:
# Confirm that Colab attached a GPU. Stop here if this fails.
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU found. In Colab select Runtime → Change runtime type → T4 GPU.'
print(torch.cuda.get_device_name(0))

In [ ]:
# Change only this URL to your GitHub repository URL, then run this cell.
REPO_URL = 'https://github.com/YOUR_USERNAME/HinglishTL.git'
assert 'YOUR_USERNAME' not in REPO_URL, 'Replace YOUR_USERNAME with your GitHub username.'
!rm -rf /content/HinglishTL
!git clone $REPO_URL /content/HinglishTL
%cd /content/HinglishTL
!git log -1 --oneline

In [ ]:
# Install only the model-server dependencies. This may take several minutes.
!pip install -q -r backend/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Dependencies installed.')

In [ ]:
# Login is required for the gated IndicTrans2 download. The token is not written to this repository.
from huggingface_hub import notebook_login, whoami
notebook_login()
print('Logged in as:', whoami()['name'])

In [ ]:
# Launch FastAPI in the background. Models load on the warm-up request below.
import os, subprocess, sys, time
env = os.environ.copy()
env['PYTHONPATH'] = '/content/HinglishTL/src'
log_file = open('/content/HinglishTL/api.log', 'w')
api_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.server:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd='/content/HinglishTL', env=env, stdout=log_file, stderr=subprocess.STDOUT
)
time.sleep(3)
!curl --fail --silent http://127.0.0.1:7860/health
print('API process started:', api_process.pid)

In [ ]:
# Warm up every model. First run downloads checkpoints and can take several minutes.
import requests
try:
    response = requests.post(
        'http://127.0.0.1:7860/assist',
        json={'text': 'kal milte hain', 'rewrite': True, 'translate': True},
        timeout=900,
    )
    response.raise_for_status()
    print(response.json())
except Exception:
    !tail -100 /content/HinglishTL/api.log
    raise

In [ ]:
# Create a temporary public URL. Keep this runtime running while using Streamlit.
import re
tunnel_log = open('/content/HinglishTL/tunnel.log', 'w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)
public_url = None
for _ in range(30):
    time.sleep(2)
    log = open('/content/HinglishTL/tunnel.log').read()
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log)
    if match:
        public_url = match.group(0)
        break
assert public_url, 'No tunnel URL found. Run: !cat /content/HinglishTL/tunnel.log'
print('Set this value in your local .env file:')
print('SERVICE_BASE_URL=' + public_url)

## Run the frontend locally

On Windows, create `C:\stuff\HinglishTL\.env` from `.env.example`, paste the printed `SERVICE_BASE_URL`, then run `streamlit run app.py`.

The tunnel is temporary and public. Use only non-sensitive demo text and stop the Colab runtime when finished.